#  Cross Track Collaboration project as a Data Engineer

In [1]:
# Importing libraries
import pandas as pd
import json

 # Loading the CSV file
df = pd.read_csv("Nkwen_traders_sales.csv")
df.head(10)


,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType
0,NKW-0001,2/7/2026,Rice 50kg,Grains,1.0,34928.0,34928,Bank Transfer,Divine K.,Walk-in
1,NKW-0002,4/13/2026,Onions 1kg,Produce,12.0,685.0,8220,Mobile Money,Florence A.,Wholesale
2,NKW-0003,4/11/2026,Salt 1kg,Groceries,11.0,297.0,3267,Orange Money,Beatrice T.,Walk-in
3,NKW-0004,1/1/2026,Bread (loaf),Bakery,13.0,591.0,7683,Bank Transfer,Divine K.,Wholesale
4,NKW-0005,2/14/2026,Beans (White),Grains,6.0,915.0,5490,Mobile Money,Beatrice T.,Walk-in
5,NKW-0006,3/9/2026,Rice 25kg,Grains,NaN,17880.0,35760,Bank Transfer,Florence A.,Walk-in
6,NKW-0007,1/14/2026,Bread (loaf),Bakery,19.0,614.0,11666,Orange Money,Florence A.,Walk-in
7,NKW-0008,3/11/2026,Rice 50kg,Grains,4.0,37414.0,149656,Bank Transfer,Ernest M.,Wholesale
8,NKW-0009,6/26/2026,Palm Oil 1L,Oils,15.0,1394.0,20910,Cash,Divine K.,Wholesale
9,NKW-0010,1/5/2026,Tomatoes 1kg,Produce,8.0,781.0,6248,Cash,Florence A.,Walk-in


##### From the sample data displayed above, the CSV is a sales transaction dataset, not a ready-made product catalogue. Therefore, for products.json, I’ld need to create one record per unique product with the prices and category rather than simply exporting all the transactions.

In [19]:
# Inspecting the data set
df.shape     
# Tells the number of columns and rows in the from the data set

(500, 10)

In [20]:
df.info() 
# Shows the data types and the number of Missing value counts per column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TransactionID   500 non-null    object 
 1   Date            500 non-null    object 
 2   Product         500 non-null    object 
 3   Category        500 non-null    object 
 4   Quantity        500 non-null    float64
 5   UnitPrice_FCFA  500 non-null    float64
 6   TotalSale_FCFA  500 non-null    int64  
 7   PaymentMethod   500 non-null    object 
 8   SalesRep        494 non-null    object 
 9   CustomerType    500 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB


In [8]:
# Check for Categories and its counts
df["Category"].value_counts()

Category
Produce       95
Groceries     82
Grains        80
Oils          74
Household     50
Groceres      30
Dairy         27
Grain         25
Bakery        22
House hold    15
Name: count, dtype: int64

##### From above, it shows that there are some inconsistencies. For example;
##### - Groceries and Groceres
#####  should be Groceries
##### - Also Grains and Grain
##### Should be Grains
##### -Household and House hold
##### Should be Household 

In [9]:
# Fixing the categories
category_mapping = {
    "Groceres": "Groceries",
    "Grain": "Grains",
    "House hold": "Household"
}
df["Category"] = df["Category"].replace(category_mapping)
df["Category"].value_counts()

Category
Groceries    112
Grains       105
Produce       95
Oils          74
Household     65
Dairy         27
Bakery        22
Name: count, dtype: int64

After this they won't be controverses over the various categories

In [2]:
df[df["Quantity"].isnull()]

,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType
5,NKW-0006,3/9/2026,Rice 25kg,Grains,NaN,17880.0,35760,Bank Transfer,Florence A.,Walk-in
25,NKW-0026,5/21/2026,Maggi Cubes (pack),Groceries,NaN,483.0,9177,Bank Transfer,Divine K.,Walk-in
29,NKW-0030,2/12/2026,Cassava (bag),Produce,NaN,3950.0,63200,Mobile Money,Divine K.,Wholesale
187,NKW-0188,4/8/2026,Onions 1kg,Produce,NaN,666.0,10656,Orange Money,Divine K.,Wholesale
189,NKW-0190,5/22/2026,Sugar 1kg,Groceries,NaN,728.0,2184,Bank Transfer,Ernest M.,Wholesale
353,NKW-0354,4/17/2026,Plantain (bunch),Produce,NaN,2603.0,36442,NaN,Florence A.,Walk-in
392,NKW-0393,4/8/2026,Tomato Paste (tin),Groceres,NaN,377.0,4901,Mobile Money,Ernest M.,Walk-in
444,NKW-0445,6/12/2026,Detergent 1kg,Household,NaN,1804.0,5412,Orange Money,Divine K.,Wholesale


##### From the above, the category Quantity has 8 missing values

In [3]:
# Calculating the missing Quantity values by dividing its total sale by its unit price
df["Quantity_Calculated"] = (
    df["TotalSale_FCFA"] / df["UnitPrice_FCFA"]
)


In [4]:
df[df["Quantity"].isnull()][
    ["Product", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "Quantity_Calculated"]
]


,Product,Quantity,UnitPrice_FCFA,TotalSale_FCFA,Quantity_Calculated
5,Rice 25kg,NaN,17880.0,35760,2.0
25,Maggi Cubes (pack),NaN,483.0,9177,19.0
29,Cassava (bag),NaN,3950.0,63200,16.0
187,Onions 1kg,NaN,666.0,10656,16.0
189,Sugar 1kg,NaN,728.0,2184,3.0
353,Plantain (bunch),NaN,2603.0,36442,14.0
392,Tomato Paste (tin),NaN,377.0,4901,13.0
444,Detergent 1kg,NaN,1804.0,5412,3.0


In [5]:
df["Quantity"] = df["Quantity"].fillna(df["Quantity_Calculated"])

In [6]:
df["Quantity"].isnull().sum()

np.int64(0)

In [23]:
# Missing Unit Price
df[df["UnitPrice_FCFA"].isnull()]

,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType


In [24]:
df["CalculatedUnitPrice"] = (
    df["TotalSale_FCFA"] / df["Quantity"]
)

In [25]:
df[df["UnitPrice_FCFA"].isnull()][
    ["Product", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "CalculatedUnitPrice"]
]

,Product,Quantity,UnitPrice_FCFA,TotalSale_FCFA,CalculatedUnitPrice


##### The unit price has 5 missing values

In [26]:
df["UnitPrice_FCFA"] = df["UnitPrice_FCFA"].fillna(df["CalculatedUnitPrice"])

In [27]:
# Checking
df["CalculatedTotal"] = (
    df["Quantity"] * df["UnitPrice_FCFA"]
)
df["SaleDifference"] = (
    df["TotalSale_FCFA"] - df["CalculatedTotal"]
)
df["SaleDifference"].abs().sum()

np.float64(0.0)

In [28]:
# Removing temporary questions
df.drop(
    columns = ["Quantity_Calculated", "CalculatedUnitPrice", "CalculatedTotal", "SaleDifference"],
    errors = "ignore",
    inplace = True
)

In [14]:
# Checking for impossible values
df["Quantity"].describe()

count    500.000000
mean      10.988000
std       10.220802
min        1.000000
25%        5.000000
50%       11.000000
75%       15.000000
max      200.000000
Name: Quantity, dtype: float64

##### From the above, the columns Quantity, UnitPrice and TotalSale have no values less then 0

##### The results show that, there are no longer missing values in the data set.

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   TransactionID   500 non-null    object        
 1   Date            500 non-null    datetime64[ns]
 2   Product         500 non-null    object        
 3   Category        500 non-null    object        
 4   Quantity        500 non-null    float64       
 5   UnitPrice_FCFA  500 non-null    float64       
 6   TotalSale_FCFA  500 non-null    float64       
 7   PaymentMethod   500 non-null    object        
 8   SalesRep        500 non-null    object        
 9   CustomerType    500 non-null    object        
dtypes: datetime64[ns](1), float64(3), object(6)
memory usage: 39.2+ KB


##### The results show that, there are no longer missing values in the data set and data types of date has been changes successfully.

In [30]:
# Counting the unique products
df["Product"].nunique()

20

In [37]:
# Creating the product catalog
df_sorted = df.sort_values("UnitPrice_FCFA")
products = (
    df_sorted.drop_duplicates("Product", keep = "last")   # Keeps the details of only the last in the sorted list
    [["Product", "Category", "UnitPrice_FCFA"]].copy()
)

In [38]:
products = products.rename(columns = {
    "Product": "Name",
    "Category": "Category",
    "UnitPrice_FCFA": "Price"
})
products

,Name,Category,Price
266,Matches (box),Household,108.0
206,Salt 1kg,Groceres,320.0
288,Tomato Paste (tin),Groceries,378.0
238,Soap (bar),House hold,422.0
437,Maggi Cubes (pack),Groceries,535.0
177,Bread (loaf),Bakery,648.0
185,Onions 1kg,Produce,748.0
64,Sugar 1kg,Groceries,800.0
107,Tomatoes 1kg,Produce,859.0
4,Beans (White),Grains,915.0


In [96]:
# Exporting products.json

In [41]:
# Creating the notebook
products.to_json( "products.json", orient = "records", indent = 4)

## Conclusions
##### The raw Nkwen Traders sales dataset was inspected and cleaned before being prepared for use by the website.

##### The cleaning process include;
##### - Checking the dataset structure and data type.
##### - Identifying and handling missing values.
##### - Standardizing inconsistent category names.
##### - Checking for duplicate transactions
##### - Validating the relationship between, quantity, unit price and total sale
##### - Creating a unique product.json
##### - Exporting the catalog